# Benchmark analysis: from MTL5's data to publication figures

MTL5 commits every benchmark run it publishes under `benchmarks/data`. Each
result file comes with a `.sysinfo` sidecar: the machine, the build, the
commit, and the machine's state (governor, clocks, pinning). `mtl5.bench`
reads those files into a DuckDB database, and keeps the provenance that
decides when two measurements are comparable. From that database it draws
figures and tables, and follows each measurement over time.

This notebook takes a sample of MTL5's own committed data through the
whole path:

1. ingest, and see which files are trusted for what;
2. one machine: its capability card, its roofline, thread scaling;
3. across machines: a table, the machine balance, a comparison;
4. over time: a trend with confidence intervals, a series break and change
   points;
5. publication: vector files, and a figure redrawn from its saved data.

The sample is in `data/mtl5-bench` (see `data/README.md`). To analyse a full
MTL5 checkout instead, set `MTL5_BENCH_DATA` to its `benchmarks/data`
directory before starting the kernel.

> Run with `pip install -e '.[notebooks]'`, which includes the `bench`
> dependencies (DuckDB, pandas, matplotlib).

In [ ]:
import os
import tempfile
from pathlib import Path

from mtl5.bench import analyze, ingest, plot, tables, trends

DATA = Path(os.environ.get("MTL5_BENCH_DATA", "data/mtl5-bench"))
work = Path(tempfile.mkdtemp(prefix="mtl5-bench-"))
DB = str(work / "bench.duckdb")

# The sample is not a git checkout of MTL5, so files are dated by their
# directory names only (a full MTL5 checkout can also date the others by git).
con, report = ingest(DATA, DB, time_from_git=False)
print(f"{len(report.outcomes)} files:", report.counts())

## 1. Ingest: every file is accounted for

The ingest records **every** file under the root, with a status and a reason.
A result file goes into one of two provenance tiers:

- **complete**: its sidecar has every required key, the build was clean, and
  it has a run time. Only complete runs enter trends and cross-machine
  comparisons.
- **incomplete**: it was read, but its provenance has gaps. Figures read it
  only when asked to, and their footer says so.

The sample's BLIS sweep predates MTL5's sidecar rules, so it's incomplete.
Here are its reasons:

In [ ]:
con.sql("""
    SELECT path, schema, tier, reasons
    FROM current_runs
    ORDER BY tier, path
""").df().tail(4)

In [ ]:
con.close()  # the analyses below open the database file themselves

## 2. One machine

### Capability card

What the machine can do, as measured by the run itself. That covers the FMA
peaks (the hardware's, and what this build reaches), the memory bandwidths,
the last-level cache, and the roofline those make. The probes are per run:
an 8-thread run reports the 8-core peak.

Every figure is a `BenchFigure`. It holds the matplotlib figure and the
DataFrame it was drawn from, and a notebook displays it.

In [ ]:
plot.capability_card(analyze.capability(DB, "i7-12700k", threads=8))

### Roofline

Every entry of one run, at its arithmetic intensity (FLOP per byte moved) and
its rate, under the machine's bandwidth slope and FMA ceilings. A point near
the slope is bandwidth-bound; a point near a ceiling is compute-bound.
`label_entries` names entries to label.

In [ ]:
plot.roofline(
    analyze.roofline(DB, "i7-12700k", threads=1),
    label_entries=["gemm/f64", "dot/f64", "spmv"],
)

### Thread scaling

Speedup over the 1-thread run, against the ideal line. The 1-thread and
8-thread runs must share the machine, the build, the commit and the machine
state; only the cores they were pinned to may differ. Bandwidth-bound
kernels (`dot`, `axpy`) saturate memory long before 8 threads; GEMM scales.

In [ ]:
scaling = analyze.thread_scaling(
    DB,
    "i7-12700k",
    entries=["dot/f64", "axpy/f64", "gemv/f64", "gemm/f64", "gemm/f32", "spmv/f64"],
)
plot.thread_scaling(scaling)

In [ ]:
scaling[scaling["threads"] == 8][["entry_id", "speedup", "efficiency"]].round(2)

## 3. Across machines

Each machine contributes its **comparable runs**: the sessions at its latest
commit, at the most threads it has a run of. MTL5 takes two sessions per
commit, so every number below is a median, and its spread is shown with it.
One session never stands in for the machine.

### The machine table

In [ ]:
machines = analyze.machines(DB)
print(tables.to_markdown(tables.machine_table(machines), footer=plot.provenance(machines)))

### Machine balance

FMA peak against memory bandwidth. A diagonal is a constant arithmetic
intensity. A machine on the `4 FLOP/B` line needs a kernel to do 4 FLOP per
byte moved before its peak, not its memory, limits it: that is its ridge
point. The x86 desktop sits far up and to the right of the two ARM boards.

In [ ]:
plot.machine_balance(machines)

### The same entries on each machine

GFLOP/s on a log axis, so a factor reads the same everywhere. A bar runs
from the least to the most the machine's sessions measured. The table gives
the numbers, as `median (min–max)`.

In [ ]:
entries = ["dot/f64", "gemv/f64", "gemm/f64", "gemm/f32", "spmv/f64", "lu/f64"]
compare = analyze.cross_machine(DB, entries)
print(
    tables.to_markdown(tables.compare_table(compare, spread=True), footer=plot.provenance(compare))
)
plot.cross_machine(compare)

As a percentage of each machine's roofline, the same entries show how close
each comes to its own bound. A hollow marker is a result MTL5 itself
flagged:

- `cache_resident`: the data fit in cache, so the DRAM roofline does not
  bound it;
- `suspect_model`: a bandwidth-bound result above 150% of its roofline,
  although its data did not fit in cache. MTL5 then doubts its own count of
  the bytes the kernel moves.

The i7's 8-thread SpMV, near 285%, is flagged `suspect_model`. The number
says more about the byte model than about the machine, which is why it's
drawn hollow rather than plotted as a result.

In [ ]:
plot.cross_machine(analyze.cross_machine(DB, entries, metric="roofline_pct"))

### Rate against problem size

The older `bench_all` sweeps measure an operation over a range of sizes.
Every sweep committed so far predates the sidecar rules, so it's in the
incomplete tier and must be asked for. The footer says
`INCOMPLETE PROVENANCE`, so the figure can't be mistaken for a comparable
measurement.

In [ ]:
plot.rate_vs_size(
    analyze.rate_vs_size(DB, operations=["gemm", "gemv", "dot"], tiers=("complete", "incomplete"))
)

## 4. Over time: a trend report

A trend follows one entry on one machine, at one label and thread count, and
reads complete, checked runs only.

- **Series.** Runs are grouped by their comparability key: the machine, its
  state, the backend, the build without the commit, the threads, and the
  entry's definition. When the key changes, for example because the build
  changed, that is a **series break**. Nothing is compared across it,
  because the two sides measure different builds.
- **Points.** Each point is one commit's runs, with a 95% confidence
  interval from the run-to-run noise between sessions.
- **Change points.** A step is flagged only if it is at least 4 standard
  errors of that noise and at least 2% in size, so a regression has to stand
  out from the noise to count.

The Ventuno Q's `native-fast` history covers three commits with two sessions
each. From `bf315e4` on, the native-architecture build names its target
explicitly (`-march=armv8.2-a+dotprod+fp16+rcpc -mtune=cortex-a78c`), and
the build's ISA gains DOTPROD. That is a different build, so it starts a new
series: a real series break.

In [ ]:
history = analyze.trend(
    DB, "arduino-ventuno-q", ["gemm/f64", "dot/f64", "spmv/f64"], label="native-fast"
)
trends.summary(history).round(3)

In [ ]:
changes = trends.change_points(history)
if changes.empty:
    print("no change points: every step is within the noise")
else:
    print(changes[["entry_id", "before", "after", "ratio", "kind"]])
plot.trend(history)

The intervals of the last commit are wide because its single pair of sessions
is all the series has after the break. That one degree of freedom gives a
Student t of 12.7. The width is the honest uncertainty, not a plotting
artefact.

The command line gives the same report, and can gate CI. This exits 3 if a
regression is found:

```
mtl5-bench trend gemm/f64 --machine arduino-ventuno-q --label native-fast --fail-on-regression
```

## 5. Publication

`save` writes vector PDF (with TrueType text) and SVG, a 300 dpi PNG, and the
DataFrame the figure was drawn from, as CSV. The files carry no timestamps,
so the same frame always gives the same bytes. A figure can therefore be
redrawn from its CSV alone, without the database, and checked against the
published one:

In [ ]:
fig = plot.cross_machine(compare)
paths = fig.save(work / "compare")
print("\n".join(p.name for p in paths))

again = plot.from_csv(work / "compare.csv", "compare")
again.save(work / "again", data=False)
same = all(
    (work / f"compare.{ext}").read_bytes() == (work / f"again.{ext}").read_bytes()
    for ext in ("pdf", "svg", "png")
)
print("redrawn from the CSV, byte for byte:", same)
assert same

Each figure's footer is its provenance: the machines, the commits, the
number of runs and their dates. A figure built from incomplete runs says so.

## From the command line

Every step above has a command:

```
mtl5-bench ingest <mtl5>/benchmarks/data --db bench.duckdb
mtl5-bench figure card --machine i7-12700k --threads 8
mtl5-bench figure roofline --machine i7-12700k --entry gemm/f64
mtl5-bench figure scaling --machine i7-12700k
mtl5-bench table machines
mtl5-bench table compare --entry gemm/f64 --entry dot/f64 --spread
mtl5-bench figure balance
mtl5-bench figure compare --metric roofline_pct
mtl5-bench figure rate --operation gemm --include-incomplete
mtl5-bench trend gemm/f64 --machine arduino-ventuno-q --label native-fast
```

The API reference documents each function under **Benchmark data**.